### Imports

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

### Hyperparams

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model Params
DATA_DIR = "/kaggle/input/datasets/olafkrastovski/handwritten-digits-0-9"
BATCH_SIZE = 64
SPLIT_SIZE = 0.8
NUM_CLASSES = 10
LEARNING_RATE = 1e-3
EPOCHS = 15

# Fuzzy Params
CLUSTERS = 4     # number of clusters
ITER = 6         # number of iterations of cluster finding
M = 2.0          # fuzziness co-efficient: controls how fuzzy membership will be
EPS = 1e-8       # small positive values to prevent division by zero

### Data Loading

In [3]:
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
])

full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)

train_size = int(SPLIT_SIZE * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Classes: {full_dataset.classes}")
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

Classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
Train: 17244, Val: 4311


### Fuzzy C-Means Layer

In [4]:
class FCMLayer(nn.Module):
    """
    Applies Fuzzy C-Means to spatial feature vectors of a conv feature map.
    Input:  (B, C, H, W)
    Output: attention map A of shape (B, 1, H, W)
    """
    def __init__(self, n_clusters=CLUSTERS, n_iter=ITER, fuzziness=M, eps=EPS):
        super().__init__()
        self.K = n_clusters
        self.n_iter = n_iter
        self.m = fuzziness
        self.eps = eps

    def forward(self, x):
        B, C, H, W = x.shape
        N = H * W
        # (B, N, C) spatial feature vectors
        X = x.permute(0, 2, 3, 1).reshape(B, N, C)

        # init centers: pick K random spatial vectors per sample
        idx = torch.stack([torch.randperm(N, device=x.device)[:self.K] for _ in range(B)])
        centers = torch.gather(X, 1, idx.unsqueeze(-1).expand(-1, -1, C))  # (B, K, C)

        for _ in range(self.n_iter):
            # distances: (B, N, K)
            dist = torch.cdist(X, centers, p=2) + self.eps
            # membership update: u_ik = 1 / sum_j (d_ik/d_jk)^(2/(m-1))
            power = 2.0 / (self.m - 1.0)
            ratio = dist.unsqueeze(-1) / dist.unsqueeze(-2)   # (B,N,K,K)
            u = 1.0 / (ratio.pow(power).sum(dim=-1) + self.eps)  # (B,N,K)

            # center update: c_k = sum_i(u_ik^m * x_i) / sum_i(u_ik^m)
            um = u.pow(self.m)                                # (B,N,K)
            weighted_sum = torch.einsum('bnk,bnc->bkc', um, X)
            denom = um.sum(dim=1).unsqueeze(-1) + self.eps    # (B,K,1)
            centers = weighted_sum / denom

        # final membership maps (B, K, H, W)
        U = u.permute(0, 2, 1).reshape(B, self.K, H, W)
        # fuzzy attention map = max membership at each spatial location
        A, _ = U.max(dim=1, keepdim=True)  # (B,1,H,W)
        return A, U

### Model

In [5]:
class FCM_CNN(nn.Module):
    def __init__(self, n_classes=10, fcm_clusters=4, use_fcm=True):
        super().__init__()
        # FCM enable flag
        self.use_fcm = use_fcm
        # 3x3 conv -> 64 feature maps
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2)   # 28 -> 14

        # 5x5 conv -> 128 feature maps
        self.conv2 = nn.Conv2d(64, 128, kernel_size=5, padding=2)
        self.pool2 = nn.MaxPool2d(2)   # 14 -> 7

        # 7x7 conv -> 256 feature maps
        self.conv3 = nn.Conv2d(128, 256, kernel_size=7, padding=3)  # stays 7x7

        self.fcm = FCMLayer(n_clusters=fcm_clusters, n_iter=6)

        self.fc1 = nn.Linear(256 * 7 * 7, 256)
        self.fc2 = nn.Linear(256, n_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        x = F.relu(self.conv2(x))
        x = self.pool2(x)

        x = F.relu(self.conv3(x))          # (B, 256, 7, 7)

        #A, _ = self.fcm(x)                 # (B, 1, 7, 7) fuzzy attention map
        #x_refined = x * A                  # element-wise refinement (Sec. 7 of PDF)
        if self.use_fcm:
            A, _ = self.fcm(x)
            x_refined = x * A
        else:
            x_refined = x
        x = x_refined.flatten(1)           # (B, 256*7*7)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

### Training Loop (Standard)

In [6]:
model = FCM_CNN(n_classes=10, fcm_clusters=4, use_fcm=False).to(device)

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

def evaluate(loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

n_epochs = EPOCHS
for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

    train_loss = running_loss / len(train_ds)
    val_acc = evaluate(val_loader)
    print(f"Epoch {epoch+1}/{n_epochs} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")

Epoch 1/15 | Loss: 1.6454 | Val Acc: 0.6799
Epoch 2/15 | Loss: 0.6601 | Val Acc: 0.8659
Epoch 3/15 | Loss: 0.3291 | Val Acc: 0.9265
Epoch 4/15 | Loss: 0.2277 | Val Acc: 0.9443
Epoch 5/15 | Loss: 0.1724 | Val Acc: 0.9362
Epoch 6/15 | Loss: 0.1347 | Val Acc: 0.9569
Epoch 7/15 | Loss: 0.1143 | Val Acc: 0.9661
Epoch 8/15 | Loss: 0.0939 | Val Acc: 0.9696
Epoch 9/15 | Loss: 0.0842 | Val Acc: 0.9717
Epoch 10/15 | Loss: 0.0677 | Val Acc: 0.9524
Epoch 11/15 | Loss: 0.0571 | Val Acc: 0.9691
Epoch 12/15 | Loss: 0.0664 | Val Acc: 0.9705
Epoch 13/15 | Loss: 0.0594 | Val Acc: 0.9701
Epoch 14/15 | Loss: 0.0397 | Val Acc: 0.9652
Epoch 15/15 | Loss: 0.0489 | Val Acc: 0.9743


### Training Loop (FCM)

In [8]:
model = FCM_CNN(n_classes=10, fcm_clusters=4, use_fcm=True).to(device)

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

def evaluate(loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

n_epochs = EPOCHS
for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

    train_loss = running_loss / len(train_ds)
    val_acc = evaluate(val_loader)
    print(f"Epoch {epoch+1}/{n_epochs} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")

Epoch 1/15 | Loss: 2.2923 | Val Acc: 0.1290
Epoch 2/15 | Loss: 2.1943 | Val Acc: 0.2526
Epoch 3/15 | Loss: 2.0560 | Val Acc: 0.2668
Epoch 4/15 | Loss: 1.9378 | Val Acc: 0.4860
Epoch 5/15 | Loss: 1.2527 | Val Acc: 0.7425
Epoch 6/15 | Loss: 0.7509 | Val Acc: 0.8432
Epoch 7/15 | Loss: 0.5224 | Val Acc: 0.8669
Epoch 8/15 | Loss: 0.4201 | Val Acc: 0.9033
Epoch 9/15 | Loss: 0.3495 | Val Acc: 0.9237
Epoch 10/15 | Loss: 0.2883 | Val Acc: 0.9186
Epoch 11/15 | Loss: 0.2520 | Val Acc: 0.9323
Epoch 12/15 | Loss: 0.2302 | Val Acc: 0.9179
Epoch 13/15 | Loss: 0.2058 | Val Acc: 0.9420
Epoch 14/15 | Loss: 0.1731 | Val Acc: 0.9350
Epoch 15/15 | Loss: 0.1712 | Val Acc: 0.9439
